# Give Your AI Agent a Real Email Inbox in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/francofuji/uncorreotemporal-mcp-server/blob/main/examples/colab-email-agent.ipynb)

This notebook is the companion to the article  
[**Using Google Colab MCP with a Temporary Email API**](https://uncorreotemporal.com/blog/google-colab-mcp-temporary-email-api/)  
published on [uncorreotemporal.com](https://uncorreotemporal.com).

It demonstrates two approaches for giving an AI agent access to a real, ephemeral email inbox:

1. **MCP approach** — via the `mcp` Python client and the UnCorreoTemporal MCP server (recommended for agents)
2. **REST API approach** — direct HTTP calls with `requests` (useful for custom pipelines)

---

**What you need:**  
An API key from [uncorreotemporal.com](https://uncorreotemporal.com). The free plan includes enough requests to run this notebook.

Replace `uct_your_key_here` with your key in the configuration cell below.

## 1. Install dependencies

In [ ]:
!pip install mcp httpx requests -q

## 2. Configuration

Set your API key here. All cells below use this value.

In [ ]:
UCT_API_KEY = "uct_your_key_here"  # <-- replace with your key

MCP_URL  = "https://uncorreotemporal.com/mcp"
REST_URL = "https://uncorreotemporal.com/api/v1"

MCP_HEADERS  = {"Authorization": f"Bearer {UCT_API_KEY}"}
REST_HEADERS = {"Authorization": f"Bearer {UCT_API_KEY}"}

---

## Part A — MCP Approach

The MCP server at `https://uncorreotemporal.com/mcp` uses streamable-HTTP transport.  
The `mcp` Python package supports this transport natively — no WebSocket or SSE setup needed.

### A1. Discover available tools

Call `tools/list` to see what the server exposes. An agent can do this dynamically — no hardcoded knowledge required.

In [ ]:
import asyncio
import json
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def list_tools():
    async with streamablehttp_client(url=MCP_URL, headers=MCP_HEADERS) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            return [(t.name, t.description) for t in tools.tools]

tools = asyncio.run(list_tools())
for name, desc in tools:
    print(f"  {name}\n    {desc}\n")

### A2. Create a temporary inbox

`create_signup_inbox` provisions a live SMTP inbox with a human-readable address.  
The inbox is active immediately and expires automatically at `expires_at`.

In [ ]:
async def create_inbox(service_name: str, ttl_minutes: int = 15):
    async with streamablehttp_client(url=MCP_URL, headers=MCP_HEADERS) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(
                "create_signup_inbox",
                {"service_name": service_name, "ttl_minutes": ttl_minutes}
            )
            return json.loads(result.content[0].text)

inbox = asyncio.run(create_inbox("MyService"))
print(f"Email address : {inbox['email']}")
print(f"Inbox ID      : {inbox['inbox_id']}")
print(f"Expires at    : {inbox['expires_at']}")

### A3. Send a test email to your inbox

Use the address above to trigger a real email delivery — register on a service, call your own SMTP relay, or use the helper below to send a test message via the API.

In [ ]:
# Optional: send a test email using an external SMTP relay or your own mail server.
# This cell is intentionally left as a placeholder — replace with your actual trigger.
#
# Example: if you have a service that sends a welcome email upon signup,
# call its registration endpoint here with the inbox['email'] address above.
#
# import requests
# requests.post(
#     "https://your-service.com/api/signup",
#     json={"email": inbox["email"], "username": "testuser"}
# )

print(f"Send an email to: {inbox['email']}")
print("Then run the next cell to wait for it.")

### A4. Wait for the email to arrive

`wait_for_verification_email` blocks (via server-side polling) until a matching email arrives or the timeout is exceeded.  
You can filter by `subject_contains` or `from_contains` to avoid reading the wrong message.

In [ ]:
async def wait_for_email(inbox_id: str, timeout: int = 90):
    async with streamablehttp_client(url=MCP_URL, headers=MCP_HEADERS) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(
                "wait_for_verification_email",
                {
                    "inbox_id": inbox_id,
                    "timeout_seconds": timeout,
                    "poll_interval_seconds": 3,
                    # Optional filters:
                    # "subject_contains": "verify",
                    # "from_contains": "noreply@",
                }
            )
            return json.loads(result.content[0].text)

message = asyncio.run(wait_for_email(inbox["inbox_id"]))
print(f"Status  : {message.get('status')}")
if message.get("status") != "timeout":
    print(f"From    : {message.get('from_address')}")
    print(f"Subject : {message.get('subject')}")
    print(f"Body    : {str(message.get('body_text', ''))[:200]}...")

### A5. Extract OTP or verification link

Both extractors can receive `message_text` directly or fetch the message themselves via `inbox_id` + `message_id`.

In [ ]:
async def extract_from_message(inbox_id: str, message_id: str):
    async with streamablehttp_client(url=MCP_URL, headers=MCP_HEADERS) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()

            otp_result = await session.call_tool(
                "extract_otp_code",
                {"inbox_id": inbox_id, "message_id": message_id}
            )
            link_result = await session.call_tool(
                "extract_verification_link",
                {"inbox_id": inbox_id, "message_id": message_id}
            )

            otp  = json.loads(otp_result.content[0].text)
            link = json.loads(link_result.content[0].text)
            return otp, link

if message.get("status") != "timeout" and message.get("message_id"):
    otp, link = asyncio.run(extract_from_message(inbox["inbox_id"], message["message_id"]))
    print(f"OTP code          : {otp.get('otp_code')}")
    print(f"Verification link : {link.get('verification_link')}")
else:
    print("No message received yet — run the wait cell above first.")

---

## Part B — End-to-End in One Call

### B1. `complete_signup_flow` — the composite tool

For agents, `complete_signup_flow` replaces the four calls above with a single tool call:  
it creates the inbox, waits for the email, and extracts both OTP and verification link in one shot.

The agent does not need to know whether the service sends an OTP or a link — it checks whichever field is non-null.

> **Note:** This cell creates a *new* inbox. Use the `email` in the response to trigger a signup before the timeout expires.

In [ ]:
async def complete_flow(service_name: str, timeout: int = 90):
    async with streamablehttp_client(url=MCP_URL, headers=MCP_HEADERS) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(
                "complete_signup_flow",
                {
                    "service_name": service_name,
                    "timeout_seconds": timeout,
                    "ttl_minutes": 15,
                    # Optional:
                    # "subject_contains": "confirm",
                    # "from_contains": "noreply",
                    # "preferred_domains": ["github.com"],
                }
            )
            return json.loads(result.content[0].text)

flow = asyncio.run(complete_flow("MyService"))

print(f"Status            : {flow['status']}")
print(f"Inbox email       : {flow.get('email')}")
print(f"Verification link : {flow.get('verification_link')}")
print(f"OTP code          : {flow.get('otp_code')}")

# Agent decision logic: use whichever was found
if flow.get("verification_link"):
    print(f"\n→ Agent action: GET {flow['verification_link']}")
elif flow.get("otp_code"):
    print(f"\n→ Agent action: submit OTP {flow['otp_code']}")
else:
    print("\n→ No verification data found (timeout or no email received)")

---

## Part C — REST API Approach

If you prefer direct HTTP calls (no MCP client), the REST API gives you the same capabilities.  
This is useful for custom pipelines where you control every step.

### C1. Create inbox via REST

In [ ]:
import requests

resp = requests.post(
    f"{REST_URL}/mailboxes",
    params={"ttl_minutes": 15},
    headers=REST_HEADERS
)
resp.raise_for_status()
data = resp.json()

rest_address       = data["address"]
rest_session_token = data["session_token"]

print(f"Email address  : {rest_address}")
print(f"Session token  : {rest_session_token[:20]}...")

### C2. Poll for messages

In [ ]:
import time
import re

msg_headers = {"X-Session-Token": rest_session_token}
deadline = time.time() + 90
message = None

print(f"Polling inbox: {rest_address}")
print("Send an email to that address within 90 seconds...")

while time.time() < deadline:
    msgs = requests.get(
        f"{REST_URL}/mailboxes/{rest_address}/messages",
        headers=msg_headers
    ).json()

    if msgs:
        msg_id = msgs[0]["id"]
        full = requests.get(
            f"{REST_URL}/mailboxes/{rest_address}/messages/{msg_id}",
            headers=msg_headers
        ).json()
        message = full
        print(f"\nEmail received!")
        print(f"From    : {full.get('from_address')}")
        print(f"Subject : {full.get('subject')}")
        break

    time.sleep(3)
    print(".", end="", flush=True)

if not message:
    print("\nTimeout — no email received in 90 seconds.")

### C3. Extract OTP or verification link (manual regex)

In [ ]:
if message:
    body = message.get("body_text", "")

    # Extract verification link
    link_match = re.search(r"https?://[^\s]+(?:verify|confirm|activate)[^\s]*", body, re.IGNORECASE)
    link = link_match.group(0) if link_match else None

    # Extract OTP (4-8 digit code)
    otp_match = re.search(r"\b(\d{4,8})\b", body)
    otp = otp_match.group(1) if otp_match else None

    print(f"Verification link : {link}")
    print(f"OTP code          : {otp}")
else:
    print("No message — run the polling cell above first.")

---

## Summary

| | MCP (`complete_signup_flow`) | REST (manual) |
|---|---|---|
| Lines of code | ~10 | ~35 |
| Polling logic | Server-side | You implement |
| OTP / link extraction | Built-in tool | Manual regex |
| Session token tracking | Automatic | You manage |
| Agent-compatible | Yes (tool discovery) | Requires glue code |

Both approaches hit real SMTP infrastructure — every email that arrives here traveled across the internet via AWS SES.

The inbox expires automatically. No cleanup needed.

---

**More resources:**
- Article: [Using Google Colab MCP with a Temporary Email API](https://uncorreotemporal.com/blog/google-colab-mcp-temporary-email-api/)
- API docs: [uncorreotemporal.com/docs/api](https://uncorreotemporal.com/docs/api)
- MCP docs: [uncorreotemporal.com/docs/mcp](https://uncorreotemporal.com/docs/mcp)
- Homepage: [uncorreotemporal.com](https://uncorreotemporal.com)